In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr
import re
import matplotlib.pyplot as plt

# ==================== 配置 ====================
class Config:
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH = './data/f1_model_final_v3.pth'
    DATA_DIR = './data/'
    
    CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
    NUM_COLS = ['year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
                'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
                'Driver_Prev_Season_FL_Count', 'Is_Home_Race', 'Recent_3_Races_Avg_Pos', 
                'Performance_Trend', 'Consistency_Score']
    TARGET_COL = 'is_winner'
    
    EMB_DIM = 32
    HIDDEN_DIM = 128
    DROPOUT_RATE = 0.4
    BATCH_SIZE = 256
    LEARNING_RATE = 5e-4
    N_EPOCHS = 50
    PATIENCE = 10

def set_seeds():
    """設定隨機種子"""
    np.random.seed(Config.SEED)
    torch.manual_seed(Config.SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.SEED)

# ==================== 數據處理 ====================
def clean_string(text):
    """清理文字"""
    return re.sub(r'\s+', ' ', text).strip() if isinstance(text, str) else text

def get_country_from_gp(gp_name):
    """從GP名稱獲取國家代碼"""
    gp_map = {
        'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
        'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
        'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
        'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
        'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
        'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
    }
    if not isinstance(gp_name, str):
        return None
    for key, country in gp_map.items():
        if key in gp_name:
            return country
    return None

def load_data():
    """載入所有CSV數據"""
    d = Config.DATA_DIR
    
    # 讀取數據
    winners = pd.read_csv(d + 'winners.csv', encoding='utf-8')
    drivers = pd.read_csv(d + 'drivers_updated.csv', encoding='utf-8')
    teams = pd.read_csv(d + 'teams_updated.csv', encoding='utf-8')
    fastest_laps = pd.read_csv(d + 'fastest_laps_updated.csv', encoding='utf-8')
    
    # 清理文字欄位
    for df in [winners, drivers, teams, fastest_laps]:
        for col in df.select_dtypes(include='object'):
            df[col] = df[col].map(clean_string)
    
    # 處理年份
    winners['year'] = pd.to_datetime(winners['Date'], errors='coerce').dt.year.astype(int)
    drivers.rename(columns={'Car': 'Team'}, inplace=True)
    
    for df in [drivers, teams, fastest_laps]:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
        if 'Pos' in df.columns:
            df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
    
    # 排除Indianapolis 500
    winners = winners[~winners['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    fastest_laps = fastest_laps[~fastest_laps['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    
    return winners, drivers, teams, fastest_laps

def create_features(drivers, teams, fastest_laps):
    """創建所有特徵"""
    def_pos_drv, def_pos_team = 50, 20
    
    # 車手lag特徵
    drivers = drivers.sort_values(['Driver', 'year'])
    drivers['Prev_Year_Driver_PTS'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers['Prev_Year_Driver_Pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(def_pos_drv)
    drivers['Driver_Experience_Years'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
    
    # 車隊lag特徵
    teams = teams.sort_values(['Team', 'year'])
    teams['Prev_Year_Team_PTS'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
    teams['Prev_Year_Team_Pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(def_pos_team)
    teams['Team_Experience_Years'] = teams['year'] - teams.groupby('Team')['year'].transform('min')
    
    # 合併車隊特徵
    drivers = drivers.merge(teams[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']], 
                           on=['Team', 'year'], how='left').fillna(0)
    
    # 最快圈速特徵
    fl = fastest_laps.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
    fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
    drivers = drivers.merge(fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']], on=['Driver', 'year'], how='left').fillna(0)
    
    # Momentum特徵
    drivers['Recent_3_Races_Avg_Pos'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).mean().reset_index(0, drop=True).fillna(def_pos_drv)
    
    # 安全計算表現趨勢
    drivers['PTS_prev'] = drivers.groupby('Driver')['PTS'].shift(1)
    drivers['Performance_Trend'] = 0.0
    mask = (drivers['PTS_prev'].notna()) & (drivers['PTS_prev'] > 0)
    drivers.loc[mask, 'Performance_Trend'] = ((drivers.loc[mask, 'PTS'] - drivers.loc[mask, 'PTS_prev']) / drivers.loc[mask, 'PTS_prev']).clip(-2.0, 2.0)
    drivers.drop('PTS_prev', axis=1, inplace=True)
    
    # 一致性評分
    drivers['Pos_Rolling_Std'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).std().reset_index(0, drop=True).fillna(0.0)
    drivers['Consistency_Score'] = 1.0 / (1.0 + drivers['Pos_Rolling_Std'])
    
    # 確保數值安全
    for col in ['Recent_3_Races_Avg_Pos', 'Performance_Trend', 'Consistency_Score']:
        drivers[col] = drivers[col].replace([float('inf'), float('-inf')], 0.0).fillna(0.0)
    
    return drivers

def build_dataset(winners, drivers):
    """構建建模數據集"""
    data = []
    drivers_by_year = {y: g for y, g in drivers.groupby('year')}
    
    for _, race in winners.iterrows():
        year, gp, winner = race['year'], race['Grand Prix'], race['Winner']
        if year not in drivers_by_year:
            continue
            
        race_country = get_country_from_gp(gp)
        for _, driver in drivers_by_year[year].iterrows():
            is_home = 1 if race_country and driver['Nationality'] == race_country else 0
            
            data.append({
                'year': year, 'Grand Prix': gp, 'Driver': driver['Driver'], 
                'Team': driver['Team'], 'Nationality': driver['Nationality'],
                'Prev_Year_Driver_PTS': driver['Prev_Year_Driver_PTS'],
                'Prev_Year_Driver_Pos': driver['Prev_Year_Driver_Pos'],
                'Driver_Experience_Years': driver['Driver_Experience_Years'],
                'Prev_Year_Team_PTS': driver['Prev_Year_Team_PTS'],
                'Prev_Year_Team_Pos': driver['Prev_Year_Team_Pos'],
                'Team_Experience_Years': driver['Team_Experience_Years'],
                'Driver_Prev_Season_FL_Count': driver['Driver_Prev_Season_FL_Count'],
                'Recent_3_Races_Avg_Pos': driver['Recent_3_Races_Avg_Pos'],
                'Performance_Trend': driver['Performance_Trend'],
                'Consistency_Score': driver['Consistency_Score'],
                'Is_Home_Race': is_home,
                'is_winner': int(driver['Driver'] == winner)
            })
    
    return pd.DataFrame(data)

def preprocess_data(train_df, test_df):
    """預處理數據"""
    # 填補缺失值
    for col in Config.CAT_COLS:
        train_df[col] = train_df[col].fillna('Unknown')
        test_df[col] = test_df[col].fillna('Unknown')
    for col in Config.NUM_COLS:
        train_df[col] = train_df[col].fillna(0)
        test_df[col] = test_df[col].fillna(0)
    
    # 編碼分類特徵
    encoders, cat_dims = {}, {}
    for col in Config.CAT_COLS:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        test_df[col] = test_df[col].map(lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else len(le.classes_))
        encoders[col] = le
        cat_dims[col] = len(le.classes_) + 1
    
    # 標準化數值特徵
    scaler = StandardScaler()
    train_df[Config.NUM_COLS] = scaler.fit_transform(train_df[Config.NUM_COLS])
    test_df[Config.NUM_COLS] = scaler.transform(test_df[Config.NUM_COLS])
    
    return train_df, test_df, encoders, scaler, cat_dims

# ==================== 模型 ====================
class FocalLoss(nn.Module):
    """Focal Loss處理類別不平衡"""
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, pred, target):
        pred_sigmoid = torch.sigmoid(pred)
        target = target.view(-1, 1)
        ce_loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        p_t = pred_sigmoid * target + (1 - pred_sigmoid) * (1 - target)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * ce_loss
        return focal_loss.mean()

class F1Dataset(Dataset):
    def __init__(self, df):
        self.x_cat = df[Config.CAT_COLS].values
        self.x_num = df[Config.NUM_COLS].values
        self.y = df[Config.TARGET_COL].values
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (torch.tensor(self.x_cat[idx], dtype=torch.long),
                torch.tensor(self.x_num[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32))

class F1Model(nn.Module):
    def __init__(self, cat_dims, num_feats):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, Config.EMB_DIM) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_feats)
        self.fc = nn.Sequential(
            nn.Linear(len(cat_dims) * Config.EMB_DIM + num_feats, Config.HIDDEN_DIM),
            nn.ReLU(), nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM, Config.HIDDEN_DIM // 2),
            nn.ReLU(), nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM // 2, 1)
        )
    
    def forward(self, x_cat, x_num):
        cat_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        num_norm = self.bn_num(x_num)
        combined = torch.cat([cat_emb, num_norm], 1)
        return self.fc(combined)

def train_model(model, train_loader, test_loader):
    """訓練模型"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    criterion = FocalLoss()
    
    best_loss = float('inf')
    no_improve = 0
    train_losses, test_losses = [], []
    
    for epoch in range(Config.N_EPOCHS):
        # 訓練
        model.train()
        train_loss = 0
        for x_cat, x_num, y in train_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_cat, x_num), y.unsqueeze(1))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # 驗證
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y in test_loader:
                x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
                test_loss += criterion(model(x_cat, x_num), y.unsqueeze(1)).item()
        
        train_loss /= len(train_loader)
        test_loss /= len(test_loader)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Test: {test_loss:.4f}")
        
        # 早停
        if test_loss < best_loss:
            best_loss = test_loss
            torch.save(model.state_dict(), Config.MODEL_PATH)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= Config.PATIENCE:
                print("Early stopping")
                break
    
    # 保存訓練曲線
    plt.figure()
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses, label='Test')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig('./data/loss_curve_final.png')
    plt.close()

def evaluate_model(model, test_loader):
    """評估模型"""
    if os.path.exists(Config.MODEL_PATH):
        model.load_state_dict(torch.load(Config.MODEL_PATH, map_location=Config.DEVICE))
    
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for x_cat, x_num, y in test_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            probs = torch.sigmoid(model(x_cat, x_num))
            pred = (probs > 0.5).squeeze().int()
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy() if pred.ndim > 0 else [pred.item()])
    
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=['Not Winner', 'Winner'], zero_division=0))

# ==================== 預測界面 ====================
class F1Predictor:
    def __init__(self):
        self.model = None
        self.encoders = {}
        self.scaler = None
        self.driver_data = {}
        self.years = []
        self.gps = []
    
    def setup(self, model, encoders, scaler, drivers, winners):
        """設置預測器"""
        self.model = model
        self.encoders = encoders
        self.scaler = scaler
        
        for year, group in drivers.groupby('year'):
            self.driver_data[year] = group.to_dict('records')
        
        self.years = sorted(drivers['year'].unique())
        self.gps = sorted(winners['Grand Prix'].unique())
    
    def predict(self, year_input, gp_input):
        """預測獲勝概率"""
        try:
            year = int(year_input)
        except:
            return "Error: Invalid year"
        
        if not gp_input or year not in self.driver_data:
            return "Error: No data available"
        
        self.model.eval()
        results = []
        
        for driver_info in self.driver_data[year]:
            # 分類特徵
            cat = []
            for col in Config.CAT_COLS:
                val = str(driver_info.get(col, 'Unknown'))
                if col == 'Grand Prix':
                    val = gp_input
                
                if val in self.encoders[col].classes_:
                    encoded = self.encoders[col].transform([val])[0]
                else:
                    encoded = len(self.encoders[col].classes_)
                cat.append(encoded)
            
            # 數值特徵
            num = []
            for col in Config.NUM_COLS:
                if col == 'Is_Home_Race':
                    race_country = get_country_from_gp(gp_input)
                    driver_nationality = driver_info.get('Nationality', '')
                    val = 1.0 if race_country and driver_nationality == race_country else 0.0
                else:
                    val = float(driver_info.get(col, 0))
                num.append(val)
            
            # 預測
            x_num = torch.tensor(self.scaler.transform([num]), dtype=torch.float32).to(Config.DEVICE)
            x_cat = torch.tensor([cat], dtype=torch.long).to(Config.DEVICE)
            
            with torch.no_grad():
                prob = torch.sigmoid(self.model(x_cat, x_num)).cpu().item()
            
            results.append((driver_info.get('Driver', 'N/A'), driver_info.get('Team', 'N/A'), prob))
        
        # 排序並返回前5名
        results.sort(key=lambda x: x[2], reverse=True)
        output = f"Predictions for {gp_input}, {year}:\n"
        for i, (driver, team, prob) in enumerate(results[:5], 1):
            output += f"{i}. {driver} ({team}): {prob:.2%}\n"
        
        return output.strip()
    
    def create_interface(self):
        """創建Gradio界面"""
        with gr.Blocks(theme=gr.themes.Soft()) as demo:
            gr.Markdown("# F1 Grand Prix Winner Predictor")
            
            with gr.Row():
                year_dd = gr.Dropdown(label="Year", choices=self.years, value=self.years[-1] if self.years else None)
                gp_dd = gr.Dropdown(label="Grand Prix", choices=self.gps, value=self.gps[0] if self.gps else None)
            
            predict_btn = gr.Button("Predict Winners")
            output_tb = gr.Textbox(label="Top 5 Predictions", lines=6, interactive=False)
            
            predict_btn.click(self.predict, inputs=[year_dd, gp_dd], outputs=[output_tb])
            
            with gr.Accordion("Training Info", open=False):
                if os.path.exists("./data/loss_curve_final.png"):
                    gr.Image(value="./data/loss_curve_final.png", label="Loss Curve")
                else:
                    gr.Markdown("Loss curve not found")
        
        return demo

# ==================== 主程序 ====================
def main():
    """主執行流程"""
    os.makedirs(Config.DATA_DIR, exist_ok=True)
    set_seeds()
    
    # 數據載入和特徵工程
    winners, drivers, teams, fastest_laps = load_data()
    drivers_feat = create_features(drivers, teams, fastest_laps)
    modeling_df = build_dataset(winners, drivers_feat)
    
    # 數據分割
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    
    if len(unique_race_ids) < 2:
        train_df = test_df = modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=Config.SEED)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    # 移除race_id並預處理
    for df in [train_df, test_df]:
        if 'race_id' in df.columns:
            df.drop(columns=['race_id'], inplace=True)
    
    train_df, test_df, encoders, scaler, cat_dims = preprocess_data(train_df, test_df)
    
    # 創建數據加載器
    train_set = F1Dataset(train_df)
    test_set = F1Dataset(test_df)
    
    weights = [1. / (train_df[Config.TARGET_COL].value_counts().get(t, 1) + 1e-6) for t in train_df[Config.TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    
    train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # 模型訓練
    cat_dims_ordered = [cat_dims[c] for c in Config.CAT_COLS]
    model = F1Model(cat_dims_ordered, len(Config.NUM_COLS)).to(Config.DEVICE)
    
    if not os.path.exists(Config.MODEL_PATH):
        train_model(model, train_loader, test_loader)
    
    evaluate_model(model, test_loader)
    
    # 啟動界面
    predictor = F1Predictor()
    predictor.setup(model, encoders, scaler, drivers_feat, winners)
    demo = predictor.create_interface()
    demo.launch(share=False)

if __name__ == '__main__':
    main()

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1 | Train: 0.1359 | Test: 0.1016
Epoch 2 | Train: 0.0952 | Test: 0.0888
Epoch 3 | Train: 0.0830 | Test: 0.0877
Epoch 4 | Train: 0.0780 | Test: 0.0814
Epoch 5 | Train: 0.0737 | Test: 0.0862
Epoch 6 | Train: 0.0688 | Test: 0.0808
Epoch 7 | Train: 0.0662 | Test: 0.0801
Epoch 8 | Train: 0.0630 | Test: 0.0809
Epoch 9 | Train: 0.0612 | Test: 0.0806
Epoch 10 | Train: 0.0593 | Test: 0.0780
Epoch 11 | Train: 0.0547 | Test: 0.0823
Epoch 12 | Train: 0.0547 | Test: 0.0758
Epoch 13 | Train: 0.0530 | Test: 0.0765
Epoch 14 | Train: 0.0499 | Test: 0.0858
Epoch 15 | Train: 0.0487 | Test: 0.0836
Epoch 16 | Train: 0.0467 | Test: 0.0858
Epoch 17 | Train: 0.0454 | Test: 0.0853
Epoch 18 | Train: 0.0437 | Test: 0.0858
Epoch 19 | Train: 0.0439 | Test: 0.0855
Epoch 20 | Train: 0.0420 | Test: 0.0909
Epoch 21 | Train: 0.0390 | Test: 0.0940
Epoch 22 | Train: 0.0415 | Test: 0.0927
Early stopping
Accuracy: 0.8803245436105477
              precision    recall  f1-score   support

  Not Winner       0.99      0

c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f